# Feature Engineering

Which columns become model inputs, how they are built, and how they are prepared for each kind of model.

* **Row and history steps** (calendar parts, days since, logs, ratios, lags, rolling windows) learn nothing from the data. They are built once on the full timeline by `src.features.build_features`.
* **Learned steps** (imputing, encoding, scaling, TargetEncoder) are an unfitted pipeline from `src.features.make_preprocessor`. Later stages fit it on training rows only, inside each CV fold.

**How to use this notebook.** The cells marked *Decision* hold the feature settings. Edit them, then choose Run All. Hand written features go in `src/custom_features.py`. Raw rows are never shown here.

## Setup

Finds the project root, loads the shared spec `ml_spec.yaml`, and checks the kernel. Run this cell first.

In [1]:
import os
import sys
from pathlib import Path

# The project root is the closest folder with ml_spec.yaml. Everything runs from there.
here = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in here if (p / "ml_spec.yaml").is_file()), Path.cwd())
os.chdir(ROOT)
# The project's own code (src) comes from this folder, also in an experiment's worktree, where the editable
# install in .venv would point to the main project.
sys.path.insert(0, str(ROOT))

# Shared code from the ML project skills
sys.path.insert(0, r"/Users/ani/.claude/skills/ml-feature-engineering/scripts")
import ml_common
import nb_common
try:  # MLflow tracking, in the ML project skills: keeps a copy of each chart for the stage's run
    import tracking
    tracking.capture_figures()
except ImportError:
    tracking = None

SPEC_PATH = str(ROOT / "ml_spec.yaml")
spec = ml_common.migrate(ml_common.load(SPEC_PATH)) if os.path.exists(SPEC_PATH) else ml_common.template()
PATHS = ml_common.project_paths(spec, str(ROOT))
NOTEBOOK = os.path.join(PATHS["notebooks"], "06_features.ipynb")
STAGE = "features"

print("Project root:", ROOT)
print("Python:", sys.executable)
if (ROOT / ".venv").is_dir() and Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    print("Warning: this kernel is not the project's .venv. Choose 'Select Kernel' and pick the project kernel.")

Project root: /Users/ani/Projects/16_weather_prediction
Python: /Users/ani/Projects/16_weather_prediction/.venv/bin/python


In [2]:
import datetime as dt
import importlib
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

import split_core

sys.path.insert(0, str(ROOT))   # the project's own src package
import src.features as F
importlib.reload(F)

<module 'src.features' from '/Users/ani/Projects/16_weather_prediction/src/features.py'>

## What This Builds On

In [3]:
framing, split = spec.get("framing") or {}, spec.get("split") or {}
TASK_TYPE, TARGET = framing.get("task_type"), split.get("target")
print(f"task: {TASK_TYPE}, target: {TARGET}, split: {split.get('strategy')}, CV: {(split.get('cv') or {}).get('method')} "
      f"with {(split.get('cv') or {}).get('n_folds')} folds")
blockers = ml_common.modelling_blockers(spec)
print("Modelling may start." if not blockers else "Not ready for modelling: " + "; ".join(blockers))

task: forecasting, target: temperature_2m_max, split: time, CV: time_backtest with 4 folds
Modelling may start.


## Load the Split Sets

Training, gap (time split only), and test rows are stacked in one table, so history features of the test rows can see the rows before them. Only training and test rows are saved at the end.

In [4]:
sets = split_core.load_sets(spec, str(ROOT))
parts = [sets["train"].assign(_set="train"), sets["test"].assign(_set="test")]
if "gap" in sets:
    parts.insert(1, sets["gap"].assign(_set="gap"))
full = pd.concat(parts, ignore_index=True)
print({name: len(part) for name, part in sets.items() if name != "folds"})

{'train': 41150, 'test': 9100}


## Feature Settings

Static location columns are known at any time; city lets the model learn each city's level. Redundant columns (grid coordinates, country, timezone) are left out. Every same day weather column is on the target date, so it would leak the outcome; each is used only as a lag of 2 days.

Lags 2 and 3 of the target are persistence and the recent trend, lag 7 a weekly view, and lag 364 the seasonal anchor. Rolling means over 7 and 28 days end on the last complete day. Two custom features compare the last complete day with the day before and with its 28 day mean.

Every lag and window ends at least 2 days before the target date, the horizon set in the framing, because requests arrive while the current day is incomplete. Lag 364 is empty for 2024.

city is one hot encoded for linear models and ordinal coded for trees, fitted inside each fold. Missing lag values are imputed with the median and flagged for linear models; trees use them as missing.

*`numeric`, `categorical` (up to about 50 values: one hot for linear models, ordinal codes for trees), and `high_cardinality` (TargetEncoder) list the input columns. `date_parts`, `days_since`, `log`, `ratios`, `lags`, and `rolling` make new columns. A lag or rolling window must look back at least `min_shift_days` (the forecast horizon), and lags of the target at least `target_min_shift_days`. `exclude` leaves columns out of the inputs. `custom: True` also runs `src/custom_features.py`.*

In [5]:
# Decision
CONFIG = {
    'numeric': ['latitude', 'longitude', 'elevation', 'temp_max_change_1d', 'temp_max_vs_28d_mean'],
    'categorical': ['city'],
    'high_cardinality': [],
    'date_parts': {'date': ['dayofyear', 'month']},
    'days_since': [],
    'log': ['elevation'],
    'ratios': [],
    'lags': [
        {'column': 'temperature_2m_max', 'days': [2, 3, 7, 364], 'by': 'city'},
        {'column': 'temperature_2m_min', 'days': [2], 'by': 'city'},
        {'column': 'temperature_2m_mean', 'days': [2], 'by': 'city'},
        {'column': 'apparent_temperature_max', 'days': [2], 'by': 'city'},
        {'column': 'apparent_temperature_min', 'days': [2], 'by': 'city'},
        {'column': 'precipitation_sum', 'days': [2], 'by': 'city'},
        {'column': 'snowfall_sum', 'days': [2], 'by': 'city'},
        {'column': 'wind_speed_10m_max', 'days': [2], 'by': 'city'},
        {'column': 'wind_gusts_10m_max', 'days': [2], 'by': 'city'},
        {'column': 'wind_direction_10m_dominant', 'days': [2], 'by': 'city'},
        {'column': 'shortwave_radiation_sum', 'days': [2], 'by': 'city'},
        {'column': 'sunshine_duration', 'days': [2], 'by': 'city'},
        {'column': 'weather_code', 'days': [2], 'by': 'city'},
    ],
    'rolling': [
        {
            'column': 'temperature_2m_max',
            'windows': [7, 28],
            'stats': ['mean'],
            'shift_days': 2,
            'by': 'city',
        },
    ],
    'custom': True,
    'date_col': 'date',
    'target': 'temperature_2m_max',
    'min_shift_days': 2,
    'target_min_shift_days': 2,
    'rare_min_frequency': 0.01,
    'exclude': [],
}

## Columns Left Out

Every column of the data that is not an input, with the reason. Move a column from here into the settings above to use it.

In [6]:
# Decision
EXCLUDED = {
    'date': 'the prediction date itself; its calendar parts are used instead',
    'grid_latitude': 'redundant with latitude (Spearman 1.00)',
    'grid_longitude': 'redundant with longitude (Spearman 1.00)',
    'country': 'redundant with city',
    'timezone': "redundant with city and country (Cramer's V 1.00)",
    'rain_sum': 'same day value; its lag is left out too, almost identical to precipitation_sum (0.98)',
    'temperature_2m_min': 'same day value on the target date, known only after the outcome; used only as a lag of 2 days',
    'temperature_2m_mean': 'same day value on the target date, known only after the outcome; used only as a lag of 2 days',
    'apparent_temperature_max': 'same day value on the target date, known only after the outcome; used only as a lag of 2 days',
    'apparent_temperature_min': 'same day value on the target date, known only after the outcome; used only as a lag of 2 days',
    'precipitation_sum': 'same day value on the target date, known only after the outcome; used only as a lag of 2 days',
    'snowfall_sum': 'same day value on the target date, known only after the outcome; used only as a lag of 2 days',
    'wind_speed_10m_max': 'same day value on the target date, known only after the outcome; used only as a lag of 2 days',
    'wind_gusts_10m_max': 'same day value on the target date, known only after the outcome; used only as a lag of 2 days',
    'wind_direction_10m_dominant': 'same day value on the target date, known only after the outcome; used only as a lag of 2 days',
    'shortwave_radiation_sum': 'same day value on the target date, known only after the outcome; used only as a lag of 2 days',
    'sunshine_duration': 'same day value on the target date, known only after the outcome; used only as a lag of 2 days',
    'weather_code': 'same day value on the target date, known only after the outcome; used only as a lag of 2 days',
}

## Why Each Feature Is Used

One line per input, shown in the feature table and kept in the spec.

In [7]:
# Decision
FEATURE_REASONS = {
    'city': "category, 50 values: lets the model learn each city's level",
    'latitude': 'static: hemisphere and climate band',
    'longitude': 'static: location',
    'elevation': 'static: altitude lowers temperature',
    'log_elevation': 'log of elevation, which is strongly skewed (helps linear models)',
    'date_dayofyear': 'smooth seasonal position of the target date',
    'date_month': 'coarse season of the target date',
    'temperature_2m_max_lag_2d': "the last complete day's maximum: persistence itself",
    'temperature_2m_max_lag_3d': 'the maximum a day before that, for the recent trend',
    'temperature_2m_max_lag_7d': 'the maximum a week before the target',
    'temperature_2m_max_lag_364d': 'the maximum on the same date last year, a seasonal anchor (empty during 2024)',
    'temperature_2m_max_roll_7d_mean_shift_2d': 'mean maximum over the week ending on the last complete day',
    'temperature_2m_max_roll_28d_mean_shift_2d': 'mean maximum over the 28 days ending on the last complete day',
    'temp_max_change_1d': 'custom: day over day change on the last complete day (lag 2 minus lag 3)',
    'temp_max_vs_28d_mean': 'custom: last complete day against its 28 day mean, warmer than usual lately',
    'temperature_2m_min_lag_2d': 'temperature_2m_min on the last complete day (2 days before the target)',
    'temperature_2m_mean_lag_2d': 'temperature_2m_mean on the last complete day (2 days before the target)',
    'apparent_temperature_max_lag_2d': 'apparent_temperature_max on the last complete day (2 days before the target)',
    'apparent_temperature_min_lag_2d': 'apparent_temperature_min on the last complete day (2 days before the target)',
    'precipitation_sum_lag_2d': 'precipitation_sum on the last complete day (2 days before the target)',
    'snowfall_sum_lag_2d': 'snowfall_sum on the last complete day (2 days before the target)',
    'wind_speed_10m_max_lag_2d': 'wind_speed_10m_max on the last complete day (2 days before the target)',
    'wind_gusts_10m_max_lag_2d': 'wind_gusts_10m_max on the last complete day (2 days before the target)',
    'wind_direction_10m_dominant_lag_2d': 'wind_direction_10m_dominant on the last complete day (2 days before the target)',
    'shortwave_radiation_sum_lag_2d': 'shortwave_radiation_sum on the last complete day (2 days before the target)',
    'sunshine_duration_lag_2d': 'sunshine_duration on the last complete day (2 days before the target)',
    'weather_code_lag_2d': 'weather_code on the last complete day (2 days before the target)',
}

## Build the Features

In [8]:
listed = list(CONFIG["numeric"]) + list(CONFIG["categorical"]) + list(CONFIG["high_cardinality"])
twice = sorted({c for c in listed if listed.count(c) > 1})
assert not twice, f"These columns are in more than one of numeric, categorical, and high_cardinality: {twice}"
start = time.time()
built = F.build_features(full, CONFIG)
inputs = F.feature_columns(CONFIG)
missing_cols = [c for c in inputs if c not in built.columns]
assert not missing_cols, f"Inputs that were not built or are not in the data: {missing_cols}"
clash = sorted(set(inputs) & set(EXCLUDED))
assert not clash, f"These columns are both inputs and left out: {clash}"
train_f = built[built["_set"] == "train"].reset_index(drop=True)
test_f = built[built["_set"] == "test"].reset_index(drop=True)
print(f"Built {len(inputs)} inputs in {time.time() - start:.1f} seconds. Training rows: {len(train_f):,}, test rows: {len(test_f):,}")

Built 27 inputs in 0.3 seconds. Training rows: 41,150, test rows: 9,100


## Feature Table

Every input with how it was made, its type, missing values, and distinct values in the training rows.

In [9]:
made = F.made_columns(CONFIG)
def kind_of(c):
    return "categorical" if c in CONFIG["categorical"] else "high cardinality" if c in CONFIG["high_cardinality"] else "numeric"
table = pd.DataFrame([{
    "feature": c, "kind": kind_of(c), "step": made.get(c, ("column", c))[0], "source": made.get(c, ("column", c))[1],
    "missing %": round(train_f[c].isna().mean() * 100, 1), "distinct": train_f[c].nunique(),
    "reason": FEATURE_REASONS.get(c, ""),
} for c in inputs])
display(table)

,feature,kind,step,source,missing %,distinct,reason
0,latitude,numeric,column,latitude,0.0,50,static: hemisphere and climate band
1,longitude,numeric,column,longitude,0.0,50,static: location
2,elevation,numeric,column,elevation,0.0,42,static: altitude lowers temperature
3,temp_max_change_1d,numeric,column,temp_max_change_1d,0.4,1193,custom: day over day change on the last comple...
4,temp_max_vs_28d_mean,numeric,column,temp_max_vs_28d_mean,0.2,16279,custom: last complete day against its 28 day m...
5,date_dayofyear,numeric,date_part,date,0.0,366,smooth seasonal position of the target date
6,date_month,numeric,date_part,date,0.0,12,coarse season of the target date
7,log_elevation,numeric,log,elevation,0.0,42,"log of elevation, which is strongly skewed (he..."
8,temperature_2m_max_lag_2d,numeric,lag,temperature_2m_max,0.2,638,the last complete day's maximum: persistence i...
9,temperature_2m_max_lag_3d,numeric,lag,temperature_2m_max,0.4,638,"the maximum a day before that, for the recent ..."


## Checks

### Single Feature Strength

Each feature on its own, scored on the CV folds: for classification the AUC of the feature alone (the larger of AUC and 1 minus AUC, so 0.5 is no signal and 1.0 is perfect), for regression and forecasting the absolute Spearman correlation. Categorical features are first replaced by the target rate of their category in the fold's training rows, so an ID cannot score well by memorising rows. A real feature rarely reaches 0.9 alone; a score that high usually means it was recorded after the outcome. Flagged features are left out unless listed in `CONFIRMED_SAFE`, with the reason in the decisions below.

In [10]:
# Decision
LEAKAGE_THRESHOLD = 0.9
CONFIRMED_SAFE = [
    'temperature_2m_max_roll_7d_mean_shift_2d',
    'temperature_2m_max_lag_2d',
    'temperature_2m_mean_lag_2d',
    'temperature_2m_max_roll_28d_mean_shift_2d',
    'apparent_temperature_max_lag_2d',
    'temperature_2m_max_lag_3d',
]

In [11]:
folds = split_core.cv_splits(train_f, split)
classification = "classification" in str(TASK_TYPE)
y_all = train_f[TARGET]
rng = np.random.default_rng(0)
keep = np.sort(rng.choice(len(train_f), 300_000, replace=False)) if len(train_f) > 300_000 else np.arange(len(train_f))
kept = np.zeros(len(train_f), dtype=bool); kept[keep] = True   # large data: score on a sample of 300,000 rows


def fold_score(x, y, train_idx, valid_idx, categorical):
    if categorical:   # target rate of each category, learned on the fold's training rows only
        key = x.astype("object").where(x.notna(), "missing").astype(str)
        rates = pd.Series(y.iloc[train_idx].values, index=key.iloc[train_idx].values).groupby(level=0).mean()
        v = key.iloc[valid_idx].map(rates).fillna(y.iloc[train_idx].mean())
    else:
        v = pd.to_numeric(x.iloc[valid_idx], errors="coerce")
        v = v.fillna(v.min() - 1 if v.notna().any() else 0)   # missing values rank lowest
    yv = y.iloc[valid_idx]
    if classification:
        classes = sorted(yv.unique())
        if len(classes) < 2:
            return np.nan
        pos = classes[-1:] if len(classes) == 2 else classes
        aucs = [roc_auc_score(yv == c, v) for c in pos]
        return max(max(a, 1 - a) for a in aucs)
    return abs(pd.Series(v.values).corr(pd.Series(yv.values), method="spearman"))


rows = []
for c in inputs:
    cat = c in CONFIG["categorical"] or c in CONFIG["high_cardinality"]
    scores = [fold_score(train_f[c], y_all, tr[kept[tr]], va[kept[va]], cat) for tr, va in folds]
    rows.append({"feature": c, "score": round(float(np.nanmean(scores)), 4), "measure": "AUC" if classification else "|Spearman|"})
strength = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
strength["flagged"] = strength["score"] >= LEAKAGE_THRESHOLD
strength["confirmed safe"] = strength["feature"].isin(CONFIRMED_SAFE)
flagged = strength.loc[strength["flagged"] & ~strength["confirmed safe"], "feature"].tolist()
display(strength.head(15))
print("Left out by this check: " + (", ".join(flagged) if flagged else "none"))
FINAL = {**CONFIG, "exclude": sorted(set(CONFIG.get("exclude") or []) | set(flagged))}
final_inputs = F.feature_columns(FINAL)
print(f"Inputs after the check: {len(final_inputs)}")

,feature,score,measure,flagged,confirmed safe
0,temperature_2m_max_roll_7d_mean_shift_2d,0.9231,|Spearman|,True,True
1,temperature_2m_max_lag_2d,0.9205,|Spearman|,True,True
2,temperature_2m_mean_lag_2d,0.9163,|Spearman|,True,True
3,temperature_2m_max_roll_28d_mean_shift_2d,0.9120,|Spearman|,True,True
4,apparent_temperature_max_lag_2d,0.9096,|Spearman|,True,True
5,temperature_2m_max_lag_3d,0.9040,|Spearman|,True,True
6,temperature_2m_max_lag_7d,0.8789,|Spearman|,False,False
7,temperature_2m_max_lag_364d,0.8761,|Spearman|,False,False
8,temperature_2m_min_lag_2d,0.8671,|Spearman|,False,False
9,apparent_temperature_min_lag_2d,0.8513,|Spearman|,False,False


Left out by this check: none
Inputs after the check: 27


### Drift Between Training and Test

The population stability index (PSI) of each feature, training rows against test rows. Under 0.1 is stable, 0.1 to 0.25 is a moderate shift, and over 0.25 is a large shift worth understanding. For a time split some drift is expected (for example calendar parts); a large shift in an ordinary feature can mean its meaning changed.

In [12]:
def psi(a, b, categorical):
    if categorical:
        a, b = a.astype("object").fillna("missing").astype(str), b.astype("object").fillna("missing").astype(str)
        top = a.value_counts().index[:20]
        a, b = a.where(a.isin(top), "other"), b.where(b.isin(top), "other")
        p, q = a.value_counts(normalize=True), b.value_counts(normalize=True)
    else:
        a, b = pd.to_numeric(a, errors="coerce"), pd.to_numeric(b, errors="coerce")
        edges = np.unique(np.nanquantile(a, np.linspace(0, 1, 11)))
        if len(edges) < 3:
            return 0.0
        edges[0], edges[-1] = -np.inf, np.inf
        p = pd.cut(a, edges).value_counts(normalize=True, dropna=False)
        q = pd.cut(b, edges).value_counts(normalize=True, dropna=False)
    p, q = p.align(q, fill_value=0)
    p, q = p.clip(lower=1e-4), q.clip(lower=1e-4)
    return float(((p - q) * np.log(p / q)).sum())


drift = pd.DataFrame([{"feature": c, "psi": round(psi(train_f[c], test_f[c], kind_of(c) != "numeric"), 3)}
                      for c in final_inputs]).sort_values("psi", ascending=False).reset_index(drop=True)
drift["shift"] = pd.cut(drift["psi"], [-1, 0.1, 0.25, np.inf], labels=["stable", "moderate", "large"])
display(drift.head(10))

,feature,psi,shift
0,temperature_2m_max_lag_364d,4.260,large
1,date_dayofyear,3.824,large
2,date_month,3.799,large
3,temperature_2m_min_lag_2d,0.373,large
4,temperature_2m_mean_lag_2d,0.368,large
5,apparent_temperature_min_lag_2d,0.366,large
6,temperature_2m_max_roll_7d_mean_shift_2d,0.356,large
7,temperature_2m_max_lag_3d,0.331,large
8,temperature_2m_max_lag_2d,0.330,large
9,temperature_2m_max_lag_7d,0.328,large


### History Windows

Every lag and rolling window, how far back it looks, and the minimum it must look back. Missing values at the start of the history are expected; many more missing values in test than in training would mean the test rows cannot see their past.

In [13]:
history = []
for lag in CONFIG.get("lags") or []:
    for d in lag["days"]:
        history.append((f"{lag['column']}_lag_{d}d", lag["column"], d))
for r in CONFIG.get("rolling") or []:
    for w in r["windows"]:
        for s in r.get("stats") or ["mean"]:
            sh = int(r.get("shift_days") or 0)
            history.append((f"{r['column']}_roll_{w}d_{s}" + (f"_shift_{sh}d" if sh else ""), r["column"], sh))
if history:
    display(pd.DataFrame([{
        "feature": name, "looks back (days)": back,
        "minimum": max(CONFIG.get("min_shift_days") or 0, (CONFIG.get("target_min_shift_days") or 0) if col == TARGET else 0),
        "missing % train": round(train_f[name].isna().mean() * 100, 1), "missing % test": round(test_f[name].isna().mean() * 100, 1),
    } for name, col, back in history]))
else:
    print("No lags or rolling windows.")

,feature,looks back (days),minimum,missing % train,missing % test
0,temperature_2m_max_lag_2d,2,2,0.2,0.0
1,temperature_2m_max_lag_3d,3,2,0.4,0.0
2,temperature_2m_max_lag_7d,7,2,0.9,0.0
3,temperature_2m_max_lag_364d,364,2,44.2,0.0
4,temperature_2m_min_lag_2d,2,2,0.2,0.0
5,temperature_2m_mean_lag_2d,2,2,0.2,0.0
6,apparent_temperature_max_lag_2d,2,2,0.2,0.0
7,apparent_temperature_min_lag_2d,2,2,0.2,0.0
8,precipitation_sum_lag_2d,2,2,0.2,0.0
9,snowfall_sum_lag_2d,2,2,0.2,0.0


### Fit Inside One Fold

Both preprocessors are fitted on the training rows of the first CV fold only, then applied to its validation rows, as the modelling stages will do. The linear version must have no missing values left; the tree version may, because tree models handle them.

In [14]:
tr, va = folds[0]
fit_check = []
for kind in ("linear", "tree"):
    start = time.time()
    pre = F.make_preprocessor(kind, FINAL)
    pre.fit(train_f.loc[tr, final_inputs], train_f.loc[tr, TARGET])
    out = pre.transform(train_f.loc[va, final_inputs])
    fit_check.append({"preprocessor": kind, "inputs": len(final_inputs), "outputs": out.shape[1],
                      "missing after": int(out.isna().sum().sum()), "seconds": round(time.time() - start, 1)})
fit_check = pd.DataFrame(fit_check)
display(fit_check)
problems = []
if int(fit_check.loc[fit_check.preprocessor == "linear", "missing after"].iloc[0]):
    problems.append("The linear preprocessor leaves missing values.")
shifted = drift.loc[drift["shift"] == "large", "feature"].tolist()
calendar = [c for c in shifted if made.get(c, ("",))[0] == "date_part"]
target_history = [c for c in shifted if made.get(c, ("", ""))[0] in ("lag", "rolling") and made[c][1] == TARGET]
large = [c for c in shifted if c not in calendar and c not in target_history]
if calendar:
    print("Calendar parts shift between training and test, which a time split always causes: " + ", ".join(calendar)
          + ". Check that the test period's values (for example its months) also appear in training.")
if target_history:
    print("Lags and windows of the target shift when the level of the series changes over time, which is expected: "
          + ", ".join(target_history) + ".")
if large:
    problems.append("Large drift between training and test: " + ", ".join(large) + ".")
print("Problems: " + (" ".join(problems) if problems else "none found."))

,preprocessor,inputs,outputs,missing after,seconds
0,linear,27,96,0,0.2
1,tree,27,27,0,0.0


Calendar parts shift between training and test, which a time split always causes: date_dayofyear, date_month. Check that the test period's values (for example its months) also appear in training.
Lags and windows of the target shift when the level of the series changes over time, which is expected: temperature_2m_max_lag_364d, temperature_2m_max_roll_7d_mean_shift_2d, temperature_2m_max_lag_3d, temperature_2m_max_lag_2d, temperature_2m_max_lag_7d, temperature_2m_max_roll_28d_mean_shift_2d.
Problems: Large drift between training and test: temperature_2m_min_lag_2d, temperature_2m_mean_lag_2d, apparent_temperature_min_lag_2d, apparent_temperature_max_lag_2d.


## Save the Feature Tables

Parquet files in the processed data folder, with the keys (row ID, fold, date, entity), the target, and every input before the learned steps.

In [15]:
keys = [c for c in dict.fromkeys([split.get("row_id_col"), split_core.FOLD, split.get("date_col"), split.get("group_col"), TARGET])
        if c and c in train_f.columns and c not in final_inputs]
files = {}
for name, frame in (("train", train_f), ("test", test_f)):
    path = os.path.join(PATHS["data_processed"], f"features_{name}.parquet")
    frame[[c for c in keys if c in frame.columns] + final_inputs].to_parquet(path, index=False)
    files[name] = ml_common.relpath(path, str(ROOT))
    print(f"{files[name]}: {len(frame):,} rows, {os.path.getsize(path) / 1e6:.1f} MB")

data/processed/features_train.parquet: 41,150 rows, 1.8 MB
data/processed/features_test.parquet: 9,100 rows, 0.5 MB


## Risks, Assumptions, Open Questions, and Decisions

Record a decision with the reason for every feature in `CONFIRMED_SAFE`. `resolved_questions` closes open questions by ID.

In [16]:
# Decision
risks = [
    {
        'problem': 'The suggested inputs used the same day weather columns, and logs of them, on the target row, which would leak the outcome.',
        'resolution': 'All same day weather columns are out; each is used only as a lag of 2 days.',
    },
    {
        'problem': 'Lag 364 is empty for the first year (about 16,000 training rows).',
        'resolution': 'Accepted: trees treat it as missing, and the linear preprocessor imputes it with a missing flag.',
    },
    {
        'problem': 'The leakage check flagged six lag and rolling features with association above 0.9.',
        'resolution': 'Confirmed safe by the user: all end on the last complete day or earlier.',
    },
    {
        'problem': 'Large drift between training and test in the 2 day lags of temperature_2m_min and apparent_temperature_min.',
        'resolution': 'Expected: the test period is April to September only, while training covers every season. Watch the test score by month in the evaluation.',
    },
]

assumptions = ["The archive's values for past days do not change after they are first published."]

open_questions = []

decisions = [
    {
        'decision': 'Same day weather columns are used only as lags of 2 days; redundant location columns are left out.',
        'reason': 'The user confirmed (Q3, Q4) that they are known only from the last complete day.',
    },
    {
        'decision': 'Custom features temp_max_change_1d and temp_max_vs_28d_mean.',
        'reason': 'The user asked for them as domain features.',
    },
    {
        'decision': 'Confirmed as safe: the target lags 2 and 3, the 7 and 28 day rolling means, and the 2 day lags of temperature_2m_mean and apparent_temperature_max, despite associations above 0.9.',
        'reason': 'Each ends at least 2 days before the target date (min_shift_days = 2, enforced by the feature code); temperature is persistent, so a high association is expected. Lag 2 is the persistence forecast itself.',
    },
]

resolved_questions = [
    {
        'id': 'Q3',
        'answer': 'temperature_2m_mean is known only 2 or more days later; it is used only as a 2 day lag.',
    },
    {
        'id': 'Q4',
        'answer': 'apparent_temperature_max is known only 2 or more days later; it is used only as a 2 day lag.',
    },
]

## Write to the Spec

Replaces the `features` section of `ml_spec.yaml` with this run. Later stages read the config from there with `src.features.load_config`.

In [17]:
code_info = (spec.get("features") or {}).get("code") or {}
features_section = split_core.plain({
    "config": FINAL,
    "inputs": [{"name": r.feature, "kind": r.kind, "step": r.step, "source": r.source, "reason": r.reason}
               for r in table[table.feature.isin(final_inputs)].itertuples()],
    "excluded": [{"column": c, "reason": r} for c, r in EXCLUDED.items()]
                + [{"column": c, "reason": f"single feature score {strength.set_index('feature').score[c]} reached the leakage threshold"} for c in flagged],
    "n_inputs": len(final_inputs),
    "outputs": {r.preprocessor: r.outputs for r in fit_check.itertuples()},
    "files": files,
    "code": {"features": code_info.get("features", "src/features.py"), "custom": code_info.get("custom", "src/custom_features.py"),
             "features_hash": code_info.get("features_hash")},
    "checks": {
        "leakage_threshold": LEAKAGE_THRESHOLD,
        "strongest": strength.head(10).to_dict("records"),
        "flagged": flagged,
        "confirmed_safe": CONFIRMED_SAFE,
        "drift_large": large,
        "problems": problems,
    },
    "run_at": dt.datetime.now().replace(microsecond=0).isoformat(),
})
nb_common.write_stage(SPEC_PATH, "features", features_section, "ml-feature-engineering", assumptions, open_questions, decisions,
                      replace=True)
not_found = nb_common.resolve_questions(SPEC_PATH, resolved_questions, "ml-feature-engineering")
if not_found:
    print("These open question IDs were not found: " + ", ".join(map(str, not_found)))
nb_common.record_run(SPEC_PATH, STAGE, NOTEBOOK, "ml-feature-engineering")
print(f"Wrote the features section of ml_spec.yaml: {len(final_inputs)} inputs.")

Wrote the features section of ml_spec.yaml: 27 inputs.


## Log to MLflow

Logs this run to the project's experiment on the shared MLflow server: the settings, the metrics, the tables, and the charts above, read from what this notebook just wrote to the spec. The run is marked as the current one for this stage. When the server is not reachable, nothing is logged and the notebook still finishes.

In [18]:
MLFLOW_RUN = tracking.log_stage(SPEC_PATH, STAGE, NOTEBOOK, "ml-feature-engineering") if tracking else None

MLflow: logged the features run to ml/16-weather-prediction: http://127.0.0.1:5000/#/experiments/5/runs/fe333b7d881e492dac39bb347b58242e


## Next Step

Run `/ml-baseline-model`. It scores the current business rule and simple models on the same CV folds, using `make_preprocessor("linear", config)` fitted inside each fold. Check progress with `/ml-status`.